In [1]:

# ============================================================
# SENSEFORGE — FINAL RESEARCH & INTEGRATION
# Notebook 17 | Single-cell pipeline
#
# Existing checkpoints are never overwritten.
# Heavy training is opt-in.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import os
import sys
import re
import gc
import json
import time
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

from transformers import AutoModel, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 0. CONFIGURATION
# ============================================================

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from senseforge.core import INVENTORY, predict as rule_predict

ART = ROOT / "artifacts"
V2 = ART / "wsd_v2"
SUP = ART / "wsd_v2_supervised"
ADV = ART / "wsd_v2_advanced"
LORA = ART / "wsd_v2_lora"
HYBRID = ART / "hybrid_v2"
OUT = ART / "senseforge_final"

OUT.mkdir(parents=True, exist_ok=True)

ORIGINAL = ROOT.parent
COMI = ORIGINAL / "Comi-Lingua"
IITB = ORIGINAL / "Data_IIT-B_Hugging_Face"
SAMANANTAR = ORIGINAL / "Dataset_Samanantar"

MODEL_NAME = "xlm-roberta-base"
DEVICE = torch.device("cpu")
MAX_LENGTH = 64
SEED = 42

# Heavy training is optional and requires an explicit opt-in.
RUN_HEAVY_TRAINING = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SENSES = [
    "bank.1", "bank.2",
    "interest.1", "interest.2",
    "plant.1", "plant.2",
    "point.1", "point.2",
    "right.1", "right.2",
    "table.1", "table.2"
]

SUPPORTED = sorted({
    s.rsplit(".", 1)[0] for s in SENSES
})

LOG = {}
START_TIME = time.time()

def section(name):
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

def save_json(name, obj):
    path = OUT / name
    path.write_text(
        json.dumps(
            obj, indent=2, ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )
    return path

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f if line.strip()
        ]

section("SENSEFORGE FINAL PIPELINE")
print("Project:", ROOT)
print("Device:", DEVICE)
print("Heavy training enabled:", RUN_HEAVY_TRAINING)

# ============================================================
# 1. ARTIFACT AUDIT
# ============================================================

section("1. ARTIFACT AUDIT")

required = {
    "Train dataset": V2 / "train.jsonl",
    "Validation dataset": V2 / "validation.jsonl",
    "Test dataset": V2 / "test.jsonl",
    "Logistic Regression": SUP / "best_supervised_model.joblib",
    "Residual MLP": ADV / "advanced_residual_mlp.pt",
    "MLP scaler": ADV / "scaler.joblib",
    "LoRA adapter": LORA / "best_lora_adapter",
    "LoRA head": LORA / "best_classifier_head.pt",
    "Hybrid config": HYBRID / "hybrid_config.json"
}

missing = []

for name, path in required.items():
    found = path.exists()
    print(f"{name:25} {'FOUND' if found else 'MISSING'}")
    if not found:
        missing.append(str(path))

if missing:
    raise FileNotFoundError(
        "Required artifacts missing:\n" + "\n".join(missing)
    )

LOG["artifact_audit"] = "passed"

# ============================================================
# 2. DATASET INTEGRITY
# ============================================================

section("2. DATASET INTEGRITY")

train_rows = read_jsonl(V2 / "train.jsonl")
val_rows = read_jsonl(V2 / "validation.jsonl")
test_rows = read_jsonl(V2 / "test.jsonl")

splits = {
    "train": train_rows,
    "validation": val_rows,
    "test": test_rows
}

expected = {
    "train": 506,
    "validation": 60,
    "test": 62
}

def norm(text):
    return " ".join(text.casefold().split())

text_sets = {}
id_sets = {}

for name, rows in splits.items():
    assert len(rows) == expected[name]

    ids = []
    texts = []

    for row in rows:
        start = int(row["target_char_start"])
        end = int(row["target_char_end"])
        text = row["text_en"]
        word = row["target_word"]
        sense = row["sense_id"]

        assert 0 <= start < end <= len(text)
        assert text[start:end].strip()
        assert sense.startswith(word + ".")
        assert sense in SENSES

        ids.append(row["record_id"])
        texts.append(norm(text))

    assert len(ids) == len(set(ids))

    text_sets[name] = set(texts)
    id_sets[name] = set(ids)

    print(
        f"{name:12} records={len(rows):3} "
        f"unique_sentences={len(text_sets[name])}"
    )

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = text_sets[a] & text_sets[b]
    id_overlap = id_sets[a] & id_sets[b]

    print(f"{a}/{b} sentence overlap:", len(overlap))
    assert not overlap
    assert not id_overlap

LOG["dataset_integrity"] = "passed"
LOG["dataset_counts"] = {
    k: len(v) for k, v in splits.items()
}

# ============================================================
# 3. LOAD EXISTING CLASSIFIERS
# ============================================================

section("3. LOADING TRAINED MODELS")

logreg = joblib.load(
    SUP / "best_supervised_model.joblib"
)

class ResidualWSDClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256,
                 num_classes=12):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.residual = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.projection(x)
        return self.classifier(
            self.norm(h + self.residual(h))
        )

mlp_ckpt = torch.load(
    ADV / "advanced_residual_mlp.pt",
    map_location="cpu",
    weights_only=True
)

assert list(mlp_ckpt["senses"]) == SENSES

mlp = ResidualWSDClassifier(
    hidden_dim=mlp_ckpt["hidden_dim"],
    num_classes=len(SENSES)
)

mlp.load_state_dict(mlp_ckpt["model_state"])
mlp.eval()

scaler = joblib.load(ADV / "scaler.joblib")

print("Logistic Regression: loaded")
print("Residual MLP: loaded")

# ============================================================
# 4. LOAD TRAINED LoRA MODEL
# ============================================================

class LoRAWSDClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()

        self.encoder = encoder
        hidden = encoder.config.hidden_size

        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask,
                target_mask, class_mask=None):

        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        weights = target_mask.unsqueeze(-1)

        pooled = (
            (hidden * weights).sum(dim=1)
            / weights.sum(dim=1).clamp(min=1)
        )

        logits = self.classifier(
            self.dropout(pooled)
        )

        if class_mask is not None:
            logits = logits.masked_fill(
                ~class_mask.bool(), -1e4
            )

        return logits

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=True
)

base = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

encoder = PeftModel.from_pretrained(
    base,
    LORA / "best_lora_adapter",
    is_trainable=False
)

lora_model = LoRAWSDClassifier(
    encoder, len(SENSES)
)

head = torch.load(
    LORA / "best_classifier_head.pt",
    map_location="cpu",
    weights_only=True
)

lora_model.classifier.load_state_dict(head)
lora_model.to(DEVICE)
lora_model.eval()

print("XLM-R LoRA: loaded")
LOG["models_loaded"] = True

# ============================================================
# 5. TARGET-CENTERED TOKENIZATION
# ============================================================

section("4. TARGET-AWARE INFERENCE")

def find_target(text, word, start=None, end=None):
    if start is not None or end is not None:
        if start is None or end is None:
            raise ValueError("Both target offsets required")

        start, end = int(start), int(end)

        if not 0 <= start < end <= len(text):
            raise ValueError("Invalid target offsets")

        return start, end

    matches = list(re.finditer(
        rf"(?<!\w){re.escape(word)}(?!\w)",
        text,
        re.IGNORECASE
    ))

    if len(matches) != 1:
        raise ValueError(
            f"Found {len(matches)} occurrences of {word}. "
            "Provide target offsets."
        )

    return matches[0].span()

def encode_target(text, word, start=None, end=None,
                  max_length=64):

    start, end = find_target(
        text, word, start, end
    )

    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_offsets_mapping=True
    )

    ids = enc["input_ids"]
    offsets = enc["offset_mapping"]

    targets = [
        i for i, (a, b) in enumerate(offsets)
        if b > a and a < end and b > start
    ]

    if not targets:
        raise ValueError("Target-token alignment failed")

    capacity = max_length - 2
    first, last = targets[0], targets[-1]

    if last - first + 1 > capacity:
        raise ValueError("Target too long")

    left = max(1, first - capacity // 2)
    left = min(
        left, max(1, last - capacity + 1)
    )

    right = min(len(ids) - 1, left + capacity)

    if right - left < capacity:
        left = max(1, right - capacity)

    selected = list(range(left, right))
    target_set = set(targets)

    input_ids = (
        [ids[0]]
        + [ids[i] for i in selected]
        + [ids[-1]]
    )

    target_mask = (
        [0.0]
        + [float(i in target_set) for i in selected]
        + [0.0]
    )

    attention = [1] * len(input_ids)
    pad = max_length - len(input_ids)

    assert pad >= 0
    assert sum(target_mask) > 0

    input_ids += [tokenizer.pad_token_id] * pad
    attention += [0] * pad
    target_mask += [0.0] * pad

    allowed = [
        s.startswith(word + ".")
        for s in SENSES
    ]

    if sum(allowed) != 2:
        raise ValueError("Unsupported neural target")

    return {
        "input_ids": torch.tensor(
            [input_ids], dtype=torch.long,
            device=DEVICE
        ),
        "attention_mask": torch.tensor(
            [attention], dtype=torch.long,
            device=DEVICE
        ),
        "target_mask": torch.tensor(
            [target_mask], dtype=torch.float32,
            device=DEVICE
        ),
        "class_mask": torch.tensor(
            [allowed], dtype=torch.bool,
            device=DEVICE
        )
    }

@torch.inference_mode()
def lora_predict(text, word, start=None, end=None):
    batch = encode_target(
        text, word, start, end, MAX_LENGTH
    )

    logits = lora_model(**batch)
    probs = torch.softmax(logits, dim=-1)[0]

    confidence, index = probs.max(dim=0)

    return {
        "sense_id": SENSES[index.item()],
        "confidence": float(confidence.item())
    }

print("Target-aware inference ready")

# ============================================================
# 6. RULE-BASED FALLBACK
# ============================================================

def rule_fallback(text, word, start=None, end=None):
    try:
        return rule_predict(
            text, word, method="rule"
        )
    except ValueError as exc:
        if "Target word not present" not in str(exc):
            raise

        if start is None or end is None:
            raise

        start, end = find_target(
            text, word, start, end
        )

        modified = text[:start] + word + text[end:]

        return rule_predict(
            modified, word, method="rule"
        )

config = json.loads(
    (HYBRID / "hybrid_config.json").read_text(
        encoding="utf-8"
    )
)

THRESHOLD = float(config["threshold"])

def hybrid_predict(text, word, start=None, end=None):
    word = word.lower().strip()

    if word not in INVENTORY:
        return {
            "text": text,
            "target_word": word,
            "sense_id": None,
            "hindi_meaning": None,
            "definition": None,
            "prediction_method": "unsupported",
            "neural_confidence": None,
            "supported": False,
            "warnings": ["Target not in inventory"]
        }

    base = rule_fallback(
        text, word, start, end
    )

    label = base["sense_id"]
    method = "rule_fallback"
    confidence = None

    if word in SUPPORTED:
        neural = lora_predict(
            text, word, start, end
        )

        confidence = neural["confidence"]

        if confidence >= THRESHOLD:
            label = neural["sense_id"]
            method = "hybrid_lora"

    sense = next(
        s for s in INVENTORY[word]["senses"]
        if s["id"] == label
    )

    return {
        "text": text,
        "target_word": word,
        "sense_id": label,
        "hindi_meaning": sense["hindi"],
        "definition": sense["definition"],
        "prediction_method": method,
        "neural_confidence": confidence,
        "supported": True,
        "warnings": [],
        "alternatives": base.get("alternatives", [])
    }

print("Hybrid inference ready")
print("Threshold:", THRESHOLD)

# ============================================================
# 7. END-TO-END ENGLISH WSD SMOKE TEST
# ============================================================

section("5. END-TO-END WSD TESTS")

examples = [
    ("The bank approved the loan.", "bank", "bank.1"),
    ("The fisherman sat on the bank.", "bank", "bank.2"),
    ("The plant needs sunlight.", "plant", "plant.1"),
    ("The factory plant closed.", "plant", "plant.2"),
    ("Turn right after the bridge.", "right", "right.2"),
    ("The answer is right.", "right", "right.1"),
    ("The table contains several rows.", "table", "table.2"),
    ("She placed the book on the table.", "table", "table.1"),
    ("He hit the ball with a bat.", "bat", "bat.1"),
    ("A bat flew from the cave.", "bat", "bat.2")
]

demo_results = []

for text, word, expected_sense in examples:
    result = hybrid_predict(text, word)

    result["expected_sense"] = expected_sense
    result["correct"] = (
        result["sense_id"] == expected_sense
    )

    demo_results.append(result)

    print(
        f"{word:10} "
        f"{result['sense_id']:12} "
        f"{result['hindi_meaning']:12} "
        f"{result['prediction_method']}"
    )

demo_accuracy = sum(
    r["correct"] for r in demo_results
) / len(demo_results)

print("\nSmoke-test accuracy:", demo_accuracy)

save_json("demo_results.json", demo_results)

LOG["smoke_test_accuracy"] = demo_accuracy

# ============================================================
# 8. MULTILINGUAL DATASET AUDIT
# ============================================================

section("6. MULTILINGUAL DATASET AUDIT")

datasets_to_audit = {
    "Comi-Lingua": COMI,
    "IIT-B": IITB,
    "Samanantar": SAMANANTAR
}

dataset_audit = {}

for name, folder in datasets_to_audit.items():
    print(f"\n{name}: {folder}")

    if not folder.exists():
        print("MISSING")
        dataset_audit[name] = {"status": "missing"}
        continue

    files = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower()
        in {".csv", ".parquet", ".jsonl"}
    ]

    dataset_audit[name] = {
        "status": "found",
        "file_count": len(files),
        "sample_files": []
    }

    for path in sorted(files)[:8]:
        entry = {
            "name": str(path.relative_to(folder)),
            "size_mb": round(
                path.stat().st_size / 1024**2, 2
            )
        }

        try:
            if path.suffix.lower() == ".csv":
                sample = pd.read_csv(path, nrows=3)
            elif path.suffix.lower() == ".parquet":
                import pyarrow.parquet as pq
                schema = pq.read_schema(path)
                entry["columns"] = schema.names
                sample = None
            else:
                sample = pd.read_json(
                    path, lines=True, nrows=3
                )

            if sample is not None:
                entry["columns"] = list(sample.columns)

        except Exception as exc:
            entry["inspection_error"] = str(exc)

        dataset_audit[name]["sample_files"].append(entry)

        print(entry)

save_json("multilingual_dataset_audit.json", dataset_audit)

# ============================================================
# 9. COMI-LINGUA LANGUAGE IDENTIFICATION
# ============================================================

section("7. COMI-LINGUA LID AUDIT")

lid_train_path = COMI / "LID_train.csv"
lid_test_path = COMI / "LID_test.csv"

lid_model = None
lid_metrics = {
    "status": "not_trained",
    "reason": "Schema not yet verified"
}

if lid_train_path.exists() and lid_test_path.exists():
    lid_train = pd.read_csv(lid_train_path)
    lid_test = pd.read_csv(lid_test_path)

    print("LID train shape:", lid_train.shape)
    print("LID test shape:", lid_test.shape)
    print("Train columns:", list(lid_train.columns))
    print("Test columns:", list(lid_test.columns))

    # Avoid guessing text and label columns.
    # Candidate names must be verified against the data.
    text_candidates = [
        "text", "sentence", "utterance"
    ]
    label_candidates = [
        "label", "language", "lang", "lid"
    ]

    text_col = next(
        (c for c in lid_train.columns
         if c.lower() in text_candidates
         and c in lid_test.columns),
        None
    )

    label_col = next(
        (c for c in lid_train.columns
         if c.lower() in label_candidates
         and c in lid_test.columns),
        None
    )

    if text_col and label_col:
        print("Candidate text column:", text_col)
        print("Candidate label column:", label_col)

        train_data = lid_train[
            [text_col, label_col]
        ].dropna().copy()

        test_data = lid_test[
            [text_col, label_col]
        ].dropna().copy()

        train_data[text_col] = (
            train_data[text_col].astype(str)
        )
        test_data[text_col] = (
            test_data[text_col].astype(str)
        )

        print("Label distribution:")
        print(train_data[label_col].value_counts().head(20))

        # Only automatically train for ordinary
        # single-label sentence-level classification.
        single_label = (
            train_data[label_col].map(
                lambda x: len(str(x).split())
            ).max() == 1
        )

        overlap = (
            set(train_data[text_col].map(norm))
            & set(test_data[text_col].map(norm))
        )

        print("Exact train/test text overlap:", len(overlap))

        if single_label and not overlap:
            lid_model = make_pipeline(
                TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(2, 5),
                    min_df=2,
                    max_features=100000,
                    sublinear_tf=True
                ),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=SEED
                )
            )

            lid_model.fit(
                train_data[text_col],
                train_data[label_col].astype(str)
            )

            lid_pred = lid_model.predict(
                test_data[text_col]
            )

            lid_gold = test_data[label_col].astype(str)

            lid_metrics = {
                "status": "trained",
                "task": "sentence_level_lid",
                "text_column": text_col,
                "label_column": label_col,
                "accuracy": float(
                    accuracy_score(lid_gold, lid_pred)
                ),
                "macro_f1": float(
                    f1_score(
                        lid_gold, lid_pred,
                        average="macro",
                        zero_division=0
                    )
                ),
                "train_records": len(train_data),
                "test_records": len(test_data)
            }

            joblib.dump(
                lid_model,
                OUT / "comilingua_lid.joblib"
            )

            print("LID results:", lid_metrics)
        else:
            lid_metrics["reason"] = (
                "Token-level labels or split overlap detected; "
                "requires a task-specific adapter."
            )
    else:
        lid_metrics["reason"] = (
            "No verified sentence-level text/label schema"
        )

print("LID status:", lid_metrics["status"])
save_json("lid_metrics.json", lid_metrics)

# ============================================================
# 10. EXISTING MODEL BENCHMARK SUMMARY
# ============================================================

section("8. RESEARCH BENCHMARK SUMMARY")

comparison_path = ART / "evaluation_v2" / "model_comparison.json"

if comparison_path.exists():
    comparison = json.loads(
        comparison_path.read_text(encoding="utf-8")
    )

    summary = pd.DataFrame.from_dict(
        comparison, orient="index"
    ).sort_values(
        "macro_f1", ascending=False
    )

    display(summary)

    summary.to_csv(
        OUT / "final_model_comparison.csv"
    )

    LOG["benchmark"] = "loaded_existing_v2"
else:
    print("No existing comparison artifact found.")
    LOG["benchmark"] = "unavailable"

# ============================================================
# 11. REGRESSION TESTS
# ============================================================

section("9. REGRESSION TESTS")

checks = {}

# Test correct metadata association.
for word in SUPPORTED:
    for sense in INVENTORY[word]["senses"]:
        assert sense["id"].startswith(word + ".")
        assert sense["hindi"]
        assert sense["definition"]

checks["sense_metadata"] = "passed"

# Test multiple occurrences require offsets.
try:
    find_target("bank then bank", "bank")
    raise AssertionError("Ambiguous target accepted")
except ValueError as exc:
    if "Found 2 occurrences" not in str(exc):
        raise

checks["multiple_occurrence_handling"] = "passed"

# Explicit second occurrence.
text = "bank then bank"
start = text.rfind("bank")
end = start + 4

assert find_target(
    text, "bank", start, end
) == (start, end)

checks["explicit_target_offsets"] = "passed"

# Check selected Hindi meaning is consistent.
for result in demo_results:
    word = result["target_word"]
    sid = result["sense_id"]

    sense = next(
        s for s in INVENTORY[word]["senses"]
        if s["id"] == sid
    )

    assert result["hindi_meaning"] == sense["hindi"]
    assert result["definition"] == sense["definition"]

checks["output_consistency"] = "passed"

# Validate dataset records can be tokenized.
for name, rows in splits.items():
    failures = []

    for row in rows:
        try:
            encode_target(
                row["text_en"],
                row["target_word"],
                row["target_char_start"],
                row["target_char_end"]
            )
        except Exception as exc:
            failures.append({
                "record_id": row["record_id"],
                "error": repr(exc)
            })

    checks[f"{name}_alignment"] = {
        "checked": len(rows),
        "failures": len(failures),
        "examples": failures[:5]
    }

    print(
        f"{name} alignment: "
        f"{len(rows) - len(failures)}/{len(rows)}"
    )

    if failures:
        raise RuntimeError(
            f"{name}: target alignment failures: "
            f"{failures[:3]}"
        )

save_json("regression_checks.json", checks)

print("Regression checks completed.")

# ============================================================
# 12. OPTIONAL ADVANCED TRAINING CONFIGURATION
# ============================================================

section("10. ADVANCED TRAINING CONFIGURATION")

heavy_config = {
    "base_model": MODEL_NAME,
    "lora_rank": 16,
    "lora_alpha": 32,
    "target_modules": [
        "query", "key", "value"
    ],
    "max_length": 128,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "max_epochs": 8,
    "learning_rate": 0.0001,
    "scheduler": "cosine",
    "selection_metric": "validation_macro_f1",
    "seed": SEED,
    "existing_checkpoints_preserved": True
}

save_json("heavy_training_config.json", heavy_config)

if RUN_HEAVY_TRAINING:
    raise NotImplementedError(
        "Heavy training is intentionally gated. "
        "The rank-16 training loop requires a separate "
        "verified smoke test, optimizer configuration "
        "and checkpoint protocol. Existing models are safe."
    )
else:
    print("Heavy training skipped (opt-in).")
    print("Configuration saved for next experiment.")

# ============================================================
# 13. FINAL RESEARCH MANIFEST
# ============================================================

section("11. FINAL RESEARCH MANIFEST")

LOG["project"] = "SenseForge"
LOG["notebook"] = "17_Final_Research_and_Integration"
LOG["neural_supported_words"] = SUPPORTED
LOG["neural_sense_classes"] = len(SENSES)
LOG["hybrid_threshold"] = THRESHOLD
LOG["lid"] = lid_metrics
LOG["heavy_training_executed"] = False
LOG["duration_seconds"] = round(
    time.time() - START_TIME, 2
)

LOG["limitations"] = [
    "Neural WSD supports six ambiguous words",
    "Existing test set has already been evaluated",
    "BabelNet mappings require independent verification",
    "Softmax confidence is not calibrated",
    "Code-mixed WSD is not independently validated",
    "Translation models are not trained in this notebook",
    "Heavy LoRA v2 training is not executed",
    "Near-duplicate and source-level leakage remain unaudited"
]

manifest_path = save_json(
    "final_research_manifest.json", LOG
)

print("\n" + "=" * 65)
print("SENSEFORGE INTEGRATION PIPELINE COMPLETED")
print("=" * 65)

print("Manifest:", manifest_path)
print("Artifacts:", OUT)
print("Runtime:", LOG["duration_seconds"], "seconds")

print("\nAvailable inference function:")
print("hybrid_predict(text, word, start=None, end=None)")

print("\nExample:")
print(
    hybrid_predict(
        "The fisherman sat on the bank.",
        "bank"
    )
)



SENSEFORGE FINAL PIPELINE
Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Device: cpu
Heavy training enabled: False

1. ARTIFACT AUDIT
Train dataset             FOUND
Validation dataset        FOUND
Test dataset              FOUND
Logistic Regression       FOUND
Residual MLP              FOUND
MLP scaler                FOUND
LoRA adapter              FOUND
LoRA head                 FOUND
Hybrid config             FOUND

2. DATASET INTEGRITY
train        records=506 unique_sentences=485
validation   records= 60 unique_sentences=53
test         records= 62 unique_sentences=57
train/validation sentence overlap: 0
train/test sentence overlap: 0
validation/test sentence overlap: 0

3. LOADING TRAINED MODELS
Logistic Regression: loaded
Residual MLP: loaded
XLM-R LoRA: loaded

4. TARGET-AWARE INFERENCE
Target-aware inference ready
Hybrid inference ready
Threshold: 0.75

5. END-TO-END WSD TESTS
bank       bank.1       बैंक         hybrid_lora
bank       bank.2       किना

,accuracy,macro_precision,macro_recall,macro_f1
Logistic Regression,1.000000,1.000000,1.000000,1.000000
Residual MLP,1.000000,1.000000,1.000000,1.000000
XLM-R LoRA,0.983871,0.992424,0.988095,0.989621
Hybrid LoRA,0.983871,0.992424,0.988095,0.989621
Rule-Based,0.467742,0.483692,0.561508,0.403322
Dictionary,0.387097,0.217495,0.500000,0.292284



9. REGRESSION TESTS
train alignment: 506/506
validation alignment: 60/60
test alignment: 62/62
Regression checks completed.

10. ADVANCED TRAINING CONFIGURATION
Heavy training skipped (opt-in).
Configuration saved for next experiment.

11. FINAL RESEARCH MANIFEST

SENSEFORGE INTEGRATION PIPELINE COMPLETED
Manifest: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/senseforge_final/final_research_manifest.json
Artifacts: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/senseforge_final
Runtime: 5.39 seconds

Available inference function:
hybrid_predict(text, word, start=None, end=None)

Example:
{'text': 'The fisherman sat on the bank.', 'target_word': 'bank', 'sense_id': 'bank.2', 'hindi_meaning': 'किनारा', 'definition': 'river shore water stream nadi नदी', 'prediction_method': 'hybrid_lora', 'neural_confidence': 0.9999221563339233, 'supported': True, 'warnings': [], 'alternatives': [{'id': 'bank.1', 'hindi': 'बैंक', 'definition': 'financial 

In [3]:
import pandas as pd

train_path = (
    ROOT.parent / "Comi-Lingua" / "LID_train.csv"
)

df = pd.read_csv(train_path, nrows=8)

print("=== COMI-LINGUA ANNOTATION INSPECTION ===")
print("Columns:", list(df.columns))

for i, row in df.iterrows():
    print(f"\n--- EXAMPLE {i + 1} ---")
    print("Sentence:", str(row["Sentences"])[:250])
    print("Predicted tags:", str(row["Predicted tags"])[:250])

    for col in df.columns:
        if "Annotator" in col:
            print(
                col + ":",
                str(row[col])[:250]
            )

=== COMI-LINGUA ANNOTATION INSPECTION ===
Columns: ['Sentences', 'Predicted tags', 'Annotated by: Annotator 1', 'Annotated by: Annotator 2', 'Annotated by: Annotator 3']

--- EXAMPLE 1 ---
Sentence: इस स्पोकन ट्यूटोरियल में हम usb driver संस्थापित करने के लिए कंप्यूटर और sbhs के बीच फिज़िकल कम्युनिकेशन सेटअप करना सीखेंगे|
Predicted tags: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'hi', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'hi', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'e
Annotated by: Annotator 1: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'en', 'index': 1}, {'key': 'ट्यूटोरियल', 'value': 'en', 'index': 2}, {'key': 'में', 'value': 'hi', 'index': 3}, {'key': 'हम', 'value': 'hi', 'index': 4}, {'key': 'usb', 'value': 'e
Annotated by: Annotator 2: [{'key': 'इस', 'value': 'hi', 'index': 0}, {'key': 'स्पोकन', 'value': 'en', 'index': 1}, {'key': 'ट्यूटो

In [5]:
import pandas as pd
import ast
from collections import Counter

path = ROOT.parent / "Comi-Lingua" / "LID_train.csv"

df = pd.read_csv(path)

annotator_cols = [
    c for c in df.columns
    if c.startswith("Annotated by:")
]

def parse_tags(value):
    if pd.isna(value):
        return []

    parsed = ast.literal_eval(value)

    if not isinstance(parsed, list):
        return []

    result = []

    for item in parsed:
        if isinstance(item, dict):
            result.append({
                "token": str(item.get("key", "")),
                "label": str(item.get("value", "")),
                "index": item.get("index")
            })

    return result

label_counts = Counter()
agreement = Counter()
invalid = 0
total_tokens = 0

for _, row in df.iterrows():
    annotations = [
        parse_tags(row[col])
        for col in annotator_cols
    ]

    lengths = [len(a) for a in annotations]

    if len(set(lengths)) != 1 or not lengths[0]:
        invalid += 1
        continue

    for tokens in zip(*annotations):
        labels = [t["label"] for t in tokens]

        if len({
            (t["token"], t["index"])
            for t in tokens
        }) != 1:
            invalid += 1
            continue

        counts = Counter(labels)
        majority_label, votes = counts.most_common(1)[0]

        label_counts[majority_label] += 1
        agreement[votes] += 1
        total_tokens += 1

print("=== COMI-LINGUA ANNOTATION AUDIT ===")
print("Sentences:", len(df))
print("Annotators:", len(annotator_cols))
print("Token labels:", dict(label_counts))
print("Agreement distribution:", dict(agreement))
print("Audited tokens:", total_tokens)
print("Invalid/misaligned entries:", invalid)

=== COMI-LINGUA ANNOTATION AUDIT ===
Sentences: 30487
Annotators: 3
Token labels: {'hi': 278112, 'en': 94936, 'ot': 36966, 'e': 40694, 'u': 15940, 'h': 127874}
Agreement distribution: {3: 538459, 2: 54114, 1: 1949}
Audited tokens: 594522
Invalid/misaligned entries: 587


In [7]:
from collections import Counter, defaultdict
import ast
import pandas as pd

label_examples = defaultdict(list)
label_sources = defaultdict(Counter)
mixed_formats = Counter()

for _, row in df.iterrows():
    for col in annotator_cols:
        try:
            parsed = ast.literal_eval(row[col])
        except (ValueError, SyntaxError, TypeError):
            continue

        if not isinstance(parsed, list):
            continue

        labels = []

        for item in parsed:
            if isinstance(item, dict):
                token = str(item.get("key", ""))
                label = str(item.get("value", ""))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                token = str(item[0])
                label = str(item[1])
            else:
                continue

            labels.append(label)
            label_sources[label][col] += 1

            if len(label_examples[label]) < 5:
                label_examples[label].append(token)

        formats = {
            "short" if label in {"h", "e", "u"}
            else "full"
            for label in labels
        }

        mixed_formats[
            "mixed" if len(formats) > 1
            else next(iter(formats), "empty")
        ] += 1

print("=== LABEL FORMAT AUDIT ===")

for label in sorted(label_sources):
    print(f"\nLabel: {label}")
    print("Sources:", dict(label_sources[label]))
    print("Examples:", label_examples[label])

print("\nFormat distribution:")
print(dict(mixed_formats))

=== LABEL FORMAT AUDIT ===

Label: e
Sources: {'Annotated by: Annotator 1': 41574, 'Annotated by: Annotator 2': 36518, 'Annotated by: Annotator 3': 42038}
Examples: ['The', 'press', 'release', 'from', 'about']

Label: en
Sources: {'Annotated by: Annotator 1': 103891, 'Annotated by: Annotator 2': 88627, 'Annotated by: Annotator 3': 90729}
Examples: ['स्पोकन', 'ट्यूटोरियल', 'usb', 'driver', 'कंप्यूटर']

Label: h
Sources: {'Annotated by: Annotator 1': 127397, 'Annotated by: Annotator 2': 131908, 'Annotated by: Annotator 3': 127758}
Examples: ['ka', 'naam', 'kyun', 'kiya', 'hai']

Label: hi
Sources: {'Annotated by: Annotator 1': 273308, 'Annotated by: Annotator 2': 282688, 'Annotated by: Annotator 3': 284314}
Examples: ['इस', 'में', 'हम', 'संस्थापित', 'करने']

Label: ot
Sources: {'Annotated by: Annotator 1': 34386, 'Annotated by: Annotator 2': 40316, 'Annotated by: Annotator 3': 36587}
Examples: ['.', '.', '33', '.', '33']

Label: u
Sources: {'Annotated by: Annotator 1': 16299, 'Annotated 

In [9]:
import ast
import pandas as pd
from collections import Counter

ALLOWED_LABELS = {"hi", "en", "ot", "h", "e", "u"}

def parse_annotation(value):
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

    if not isinstance(parsed, list):
        return []

    result = []

    for item in parsed:
        if isinstance(item, dict):
            token = str(item.get("key", ""))
            label = str(item.get("value", ""))
            index = item.get("index")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            token = str(item[0])
            label = str(item[1])
            index = len(result)
        else:
            return []

        result.append({
            "token": token,
            "label": label,
            "index": index
        })

    return result


token_records = []
skipped = Counter()

for sentence_id, row in df.iterrows():
    annotations = [
        parse_annotation(row[col])
        for col in annotator_cols
    ]

    if not all(annotations):
        skipped["empty_annotation"] += 1
        continue

    lengths = [len(a) for a in annotations]

    if len(set(lengths)) != 1:
        skipped["length_mismatch"] += 1
        continue

    for position, group in enumerate(zip(*annotations)):
        tokens = [x["token"] for x in group]
        labels = [x["label"] for x in group]

        if len(set(tokens)) != 1:
            skipped["token_mismatch"] += 1
            continue

        if not all(label in ALLOWED_LABELS for label in labels):
            skipped["unknown_label"] += 1
            continue

        votes = Counter(labels)
        majority_label, vote_count = votes.most_common(1)[0]

        if vote_count < 2:
            skipped["no_majority"] += 1
            continue

        token_records.append({
            "sentence_id": int(sentence_id),
            "position": position,
            "token": tokens[0],
            "label": majority_label,
            "agreement": vote_count
        })

tokens_df = pd.DataFrame(token_records)

print("=== TOKEN DATASET ===")
print("Usable tokens:", len(tokens_df))
print("Unique sentences:", tokens_df["sentence_id"].nunique())
print("Labels:", tokens_df["label"].value_counts().to_dict())
print("Skipped:", dict(skipped))

output = OUT / "comilingua_train_tokens.csv"

tokens_df.to_csv(output, index=False)

print("Saved:", output)

=== TOKEN DATASET ===
Usable tokens: 592621
Unique sentences: 30395
Labels: {'hi': 277984, 'h': 127867, 'en': 93551, 'e': 40529, 'ot': 36813, 'u': 15877}
Skipped: {'no_majority': 1950, 'token_mismatch': 493, 'empty_annotation': 4, 'length_mismatch': 84}
Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/senseforge_final/comilingua_train_tokens.csv


In [11]:
# ============================================================
# SENSEFORGE — COMI-LINGUA TOKEN-LEVEL LID
# Supervised training + held-out evaluation
# ============================================================

import ast
import json
import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

LID_OUT = OUT / "comilingua_lid"
LID_OUT.mkdir(parents=True, exist_ok=True)

LABELS = ["hi", "en", "ot", "h", "e", "u"]
LABEL_SET = set(LABELS)

# ------------------------------------------------------------
# 1. Load prepared training tokens
# ------------------------------------------------------------

train_tokens = pd.read_csv(
    OUT / "comilingua_train_tokens.csv"
)

required = {
    "sentence_id", "position",
    "token", "label", "agreement"
}

assert required.issubset(train_tokens.columns)
assert train_tokens["label"].isin(LABEL_SET).all()

train_tokens = train_tokens.dropna(
    subset=["token", "label", "sentence_id"]
).copy()

train_tokens["token"] = train_tokens["token"].astype(str)
train_tokens["label"] = train_tokens["label"].astype(str)

print("Training tokens:", len(train_tokens))
print("Training sentences:", train_tokens["sentence_id"].nunique())

# ------------------------------------------------------------
# 2. Parse original test annotations
# ------------------------------------------------------------

test_path = ROOT.parent / "Comi-Lingua" / "LID_test.csv"
test_df = pd.read_csv(test_path)

annotator_cols = [
    c for c in test_df.columns
    if c.startswith("Annotated by:")
]

assert len(annotator_cols) == 3

def parse_annotations(value):
    if pd.isna(value):
        return []

    try:
        data = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

    if not isinstance(data, list):
        return []

    result = []

    for item in data:
        if isinstance(item, dict):
            token = str(item.get("key", ""))
            label = str(item.get("value", ""))
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            token = str(item[0])
            label = str(item[1])
        else:
            return []

        result.append((token, label))

    return result


test_records = []
skipped = Counter()

for sentence_id, row in test_df.iterrows():
    annotations = [
        parse_annotations(row[col])
        for col in annotator_cols
    ]

    if not all(annotations):
        skipped["empty"] += 1
        continue

    if len(set(map(len, annotations))) != 1:
        skipped["length_mismatch"] += 1
        continue

    for position, group in enumerate(zip(*annotations)):
        tokens = [x[0] for x in group]
        labels = [x[1] for x in group]

        if len(set(tokens)) != 1:
            skipped["token_mismatch"] += 1
            continue

        if not all(x in LABEL_SET for x in labels):
            skipped["unknown_label"] += 1
            continue

        votes = Counter(labels)
        majority, count = votes.most_common(1)[0]

        if count < 2:
            skipped["no_majority"] += 1
            continue

        test_records.append({
            "sentence_id": sentence_id,
            "position": position,
            "token": tokens[0],
            "label": majority,
            "agreement": count
        })

test_tokens = pd.DataFrame(test_records)

assert not test_tokens.empty

print("Test tokens:", len(test_tokens))
print("Test sentences:", test_tokens["sentence_id"].nunique())
print("Skipped:", dict(skipped))

# ------------------------------------------------------------
# 3. Leakage audit
# ------------------------------------------------------------

original_train = pd.read_csv(
    ROOT.parent / "Comi-Lingua" / "LID_train.csv",
    usecols=["Sentences"]
)

def normalize_sentence(x):
    return " ".join(str(x).casefold().split())

train_sentences = set(
    original_train["Sentences"].dropna().map(normalize_sentence)
)

test_sentences = set(
    test_df["Sentences"].dropna().map(normalize_sentence)
)

overlap = train_sentences & test_sentences

print("Exact train/test sentence overlap:", len(overlap))

if overlap:
    raise RuntimeError(
        "Original Comi-Lingua train/test sentence overlap found. "
        "Do not evaluate until leakage is resolved."
    )

# ------------------------------------------------------------
# 4. Sentence-group validation split
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.12,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_tokens,
        groups=train_tokens["sentence_id"]
    )
)

fit_df = train_tokens.iloc[train_idx]
val_df = train_tokens.iloc[val_idx]

assert not (
    set(fit_df["sentence_id"])
    & set(val_df["sentence_id"])
)

print("Fit tokens:", len(fit_df))
print("Validation tokens:", len(val_df))

# ------------------------------------------------------------
# 5. Model training
# ------------------------------------------------------------

model_lid = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=3,
            max_features=120000,
            sublinear_tf=True,
            lowercase=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=600,
            class_weight="balanced",
            random_state=42
        )
    )
])

start_time = time.time()

model_lid.fit(
    fit_df["token"],
    fit_df["label"]
)

training_seconds = time.time() - start_time

print("\nTraining completed.")
print("Training time:", round(training_seconds, 2), "seconds")

# ------------------------------------------------------------
# 6. Validation evaluation
# ------------------------------------------------------------

val_pred = model_lid.predict(
    val_df["token"]
)

val_gold = val_df["label"]

validation_metrics = {
    "accuracy": float(
        accuracy_score(val_gold, val_pred)
    ),
    "macro_precision": float(
        precision_score(
            val_gold, val_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "macro_recall": float(
        recall_score(
            val_gold, val_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "macro_f1": float(
        f1_score(
            val_gold, val_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    )
}

print("\n=== VALIDATION METRICS ===")
print(validation_metrics)

# ------------------------------------------------------------
# 7. Original held-out test evaluation
# ------------------------------------------------------------

test_pred = model_lid.predict(
    test_tokens["token"].astype(str)
)

test_gold = test_tokens["label"]

test_metrics = {
    "accuracy": float(
        accuracy_score(test_gold, test_pred)
    ),
    "macro_precision": float(
        precision_score(
            test_gold, test_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "macro_recall": float(
        recall_score(
            test_gold, test_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    ),
    "macro_f1": float(
        f1_score(
            test_gold, test_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    )
}

print("\n=== HELD-OUT TEST METRICS ===")
print(test_metrics)

print("\nClassification report:")
print(classification_report(
    test_gold,
    test_pred,
    labels=LABELS,
    zero_division=0
))

# ------------------------------------------------------------
# 8. Save model and reports
# ------------------------------------------------------------

joblib.dump(
    model_lid,
    LID_OUT / "comilingua_token_lid.joblib"
)

metadata = {
    "model": "char_tfidf_logistic_regression",
    "task": "token_level_lid",
    "labels": LABELS,
    "train_tokens": len(fit_df),
    "validation_tokens": len(val_df),
    "test_tokens": len(test_tokens),
    "training_seconds": training_seconds,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "test_annotation_skips": dict(skipped),
    "label_semantics": "original_six_labels_unverified"
}

(LID_OUT / "lid_results.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

predictions_df = test_tokens.copy()
predictions_df["predicted_label"] = test_pred

predictions_df.to_csv(
    LID_OUT / "test_predictions.csv",
    index=False
)

print("\nSaved model and reports:", LID_OUT)

# ------------------------------------------------------------
# 9. Example predictions
# ------------------------------------------------------------

examples = [
    "hello",
    "namaste",
    "computer",
    "भारत",
    "school",
    "ka",
    "hai",
    "tutorial"
]

print("\n=== TOKEN PREDICTION DEMO ===")

for token, label in zip(
    examples,
    model_lid.predict(examples)
):
    print(f"{token:15} -> {label}")

print("\nCOMI-LINGUA LID EXPERIMENT COMPLETE")

Training tokens: 592619
Training sentences: 30395
Test tokens: 106849
Test sentences: 4995
Skipped: {'no_majority': 307, 'token_mismatch': 60, 'length_mismatch': 5}
Exact train/test sentence overlap: 4120


RuntimeError: Original Comi-Lingua train/test sentence overlap found. Do not evaluate until leakage is resolved.

In [13]:

from sklearn.model_selection import GroupShuffleSplit
from collections import Counter
import numpy as np
import pandas as pd

# Combine existing annotated token records.
# Prefix original sentence IDs to avoid collisions.
train_clean = train_tokens.copy()
test_clean = test_tokens.copy()

train_clean["source_split"] = "original_train"
test_clean["source_split"] = "original_test"

train_clean["source_sentence_id"] = (
    "train_" + train_clean["sentence_id"].astype(str)
)

test_clean["source_sentence_id"] = (
    "test_" + test_clean["sentence_id"].astype(str)
)

# Attach original sentence text.
original_train = pd.read_csv(
    ROOT.parent / "Comi-Lingua" / "LID_train.csv",
    usecols=["Sentences"]
)

original_test = pd.read_csv(
    ROOT.parent / "Comi-Lingua" / "LID_test.csv",
    usecols=["Sentences"]
)

train_clean["sentence"] = train_clean["sentence_id"].map(
    original_train["Sentences"]
)

test_clean["sentence"] = test_clean["sentence_id"].map(
    original_test["Sentences"]
)

combined = pd.concat(
    [train_clean, test_clean],
    ignore_index=True
)

combined["sentence_group"] = (
    combined["sentence"]
    .astype(str)
    .map(lambda x: " ".join(x.casefold().split()))
)

assert combined["sentence_group"].notna().all()

# Split at normalized sentence level.
groups = combined["sentence_group"].unique()

rng = np.random.default_rng(42)
rng.shuffle(groups)

n = len(groups)

n_test = int(n * 0.15)
n_val = int(n * 0.15)

test_groups = set(groups[:n_test])
val_groups = set(groups[n_test:n_test + n_val])
train_groups = set(groups[n_test + n_val:])

safe_train = combined[
    combined["sentence_group"].isin(train_groups)
].copy()

safe_val = combined[
    combined["sentence_group"].isin(val_groups)
].copy()

safe_test = combined[
    combined["sentence_group"].isin(test_groups)
].copy()

assert not (train_groups & val_groups)
assert not (train_groups & test_groups)
assert not (val_groups & test_groups)

print("=== LEAKAGE-SAFE SPLITS ===")

for name, data in [
    ("Train", safe_train),
    ("Validation", safe_val),
    ("Test", safe_test)
]:
    print(
        name,
        "tokens:", len(data),
        "sentences:", data["sentence_group"].nunique()
    )

SPLIT_OUT = OUT / "comilingua_lid" / "safe_splits"
SPLIT_OUT.mkdir(parents=True, exist_ok=True)

for name, data in [
    ("train", safe_train),
    ("validation", safe_val),
    ("test", safe_test)
]:
    data.to_parquet(
        SPLIT_OUT / f"{name}.parquet",
        index=False
    )

print("\nSaved:", SPLIT_OUT)
print("Exact sentence leakage: 0")


/var/folders/23/m5f33lv50c505pxyhh15gmym0000gn/T/ipykernel_1553/866097853.py:58: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(groups)


=== LEAKAGE-SAFE SPLITS ===
Train tokens: 489452 sentences: 21886
Validation tokens: 106144 sentences: 4689
Test tokens: 103872 sentences: 4689

Saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/senseforge_final/comilingua_lid/safe_splits
Exact sentence leakage: 0


In [15]:

from pathlib import Path
from collections import Counter
import pandas as pd

SPLIT_OUT = (
    OUT / "comilingua_lid" / "safe_splits"
)

splits = {
    name: pd.read_parquet(
        SPLIT_OUT / f"{name}.parquet"
    )
    for name in ["train", "validation", "test"]
}

print("=== SPLIT INTEGRITY AUDIT ===")

groups = {}

for name, data in splits.items():
    groups[name] = set(
        data["sentence_group"].astype(str)
    )

    print(
        f"{name:12} "
        f"tokens={len(data):7} "
        f"sentences={len(groups[name]):5}"
    )

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = groups[a] & groups[b]

    print(f"{a} vs {b}: {len(overlap)} overlaps")
    assert not overlap

combined = pd.concat(
    splits.values(),
    ignore_index=True
)

# Same sentence and token position may occur
# in multiple original source records.
key_columns = [
    "sentence_group",
    "position",
    "token"
]

grouped = combined.groupby(
    key_columns,
    dropna=False
)["label"]

unique_labels = grouped.nunique()

conflicting = unique_labels[
    unique_labels > 1
]

duplicate_counts = combined.duplicated(
    subset=key_columns,
    keep=False
)

print("\n=== ANNOTATION AUDIT ===")
print("Total token records:", len(combined))
print("Repeated token records:", int(duplicate_counts.sum()))
print("Conflicting token groups:", len(conflicting))

if len(conflicting):
    print("\nExample conflicting groups:")
    print(conflicting.head(10))

print("\nAudit complete.")


=== SPLIT INTEGRITY AUDIT ===
train        tokens= 489452 sentences=21886
validation   tokens= 106144 sentences= 4689
test         tokens= 103872 sentences= 4689
train vs validation: 0 overlaps
train vs test: 0 overlaps
validation vs test: 0 overlaps

=== ANNOTATION AUDIT ===
Total token records: 699468
Repeated token records: 174878
Conflicting token groups: 107

Example conflicting groups:
sentence_group                                                                                                                                                                  position  token       
manisha published: mwc 2025 : qualcomm technologies ने बार्सिलोना में आयोजित mwc 2025 के दौरान कई बड़े ऐलान किए हैं, जिनमें से एक dragonwing fwa gen 4 elite की अनाउंसमेंट है।  0         Manisha         2
                                                                                                                                                                                1         Published       

In [17]:

from pathlib import Path
from collections import Counter
import pandas as pd
import json

SPLIT_OUT = OUT / "comilingua_lid" / "safe_splits"
CLEAN_OUT = OUT / "comilingua_lid" / "clean_splits"
CLEAN_OUT.mkdir(parents=True, exist_ok=True)

KEY = ["sentence_group", "position", "token"]

clean_splits = {}
audit = {}

for split in ["train", "validation", "test"]:
    df_split = pd.read_parquet(
        SPLIT_OUT / f"{split}.parquet"
    )

    original_count = len(df_split)

    # Detect label disagreements for the same token identity.
    label_counts = (
        df_split.groupby(KEY, dropna=False)["label"]
        .nunique()
        .reset_index(name="unique_labels")
    )

    # Keep only token identities with one consistent label.
    consistent = label_counts[
        label_counts["unique_labels"] == 1
    ][KEY]

    clean = df_split.merge(
        consistent,
        on=KEY,
        how="inner"
    )

    # Collapse repeated annotations.
    clean = clean.drop_duplicates(
        subset=KEY,
        keep="first"
    ).copy()

    assert not clean.duplicated(KEY).any()

    # Ensure no conflicting labels remain.
    assert (
        clean.groupby(KEY, dropna=False)["label"]
        .nunique()
        .max()
    ) <= 1

    clean_splits[split] = clean

    audit[split] = {
        "original_records": original_count,
        "clean_records": len(clean),
        "removed_records": original_count - len(clean),
        "unique_sentences": clean["sentence_group"].nunique()
    }

    clean.to_parquet(
        CLEAN_OUT / f"{split}.parquet",
        index=False
    )

    print(
        f"{split:12} "
        f"original={original_count:7} "
        f"clean={len(clean):7} "
        f"removed={original_count-len(clean):7}"
    )

# Cross-split leakage audit.
groups = {
    name: set(
        data["sentence_group"].astype(str)
    )
    for name, data in clean_splits.items()
}

assert not (groups["train"] & groups["validation"])
assert not (groups["train"] & groups["test"])
assert not (groups["validation"] & groups["test"])

audit["conflict_policy"] = (
    "Exclude token identities with conflicting labels; "
    "collapse exact duplicate annotations."
)

audit["sentence_overlap"] = 0

(CLEAN_OUT / "cleaning_audit.json").write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8"
)

print("\nClean splits saved:", CLEAN_OUT)
print("Cross-split sentence leakage: 0")


train        original= 489452 clean= 428197 removed=  61255
validation   original= 106144 clean=  92552 removed=  13592
test         original= 103872 clean=  91173 removed=  12699

Clean splits saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/senseforge_final/comilingua_lid/clean_splits
Cross-split sentence leakage: 0
